<a href="https://colab.research.google.com/github/CarlosYazid/Tabu-Search-Class/blob/main/src/notebooks/Practica_Busqueda_Tabu_CVRP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Búsqueda Tabú: distribuir vacunas en Trípoli (CVRP)

Debes repartir **38.000 dosis** desde el depósito **MSO** hasta **10 centros de salud** de Trípoli. La flota tiene 15 vehículos de **6.000 unidades** de capacidad. Hay que decidir **qué centros atiende cada vehículo y en qué orden**, de modo que el **tiempo total de manejo** sea el menor posible.

| Elemento | En este problema |
|---|---|
| **Representación** | `x` = permutación de los 10 centros (un *tour gigante*). Ejemplo: `[3, 7, 1, 10, 4, 2, 9, 5, 8, 6]`. Un decodificador (*Split*) lo corta en rutas que respetan la capacidad |
| **Vecindario** | Intercambiar **dos centros consecutivos** del tour → 9 vecinos por solución |
| **Función de aptitud** | Tiempo total (min) de las rutas que resultan de decodificar `x` (menor es mejor) |
| **Atributo tabú** | `(centro, posición anterior)`: prohibido devolver ese centro a esa posición durante `TENENCIA` iteraciones |
| **Aspiración** | Un movimiento tabú se permite si produce una solución mejor que la mejor conocida |

## Cómo funciona esta actividad
1. **Completa 4 bloques `TODO`** (vecindario, `es_tabu`, `actualizar_lista_tabu` y la selección del mejor vecino con aspiración). Después de cada uno hay una celda de verificación rápida.
2. **Ejecuta tu búsqueda tabú** y revisa las gráficas: rutas en el mapa, convergencia, cargas por vehículo y el efecto de la tenencia.
3. **Checks:** el notebook evalua progresivamente tur respuestas.
4. **Analiza** tus resultados respondiendo las preguntas del final.


## Rúbrica

| Criterio | Prueba que lo evalúa | Puntos |
|---|---|---|
| Respuesta factible (depósito, visita única, capacidad, ≤ 15 vehículos) | `test_external_answer_coherence` | 1,00 |
| Tiempo total a ≤ 10 % de la referencia | `test_external_answer_quality_vs_tabu_search[10pct]` | 0,75 |
| Tiempo total a ≤ 5 % de la referencia | `test_external_answer_quality_vs_tabu_search[5pct]` | 0,75 |
| Tiempo total a ≤ 2 % de la referencia | `test_external_answer_quality_vs_tabu_search[2pct]` | 0,75 |
| Iguala o supera la referencia |  `test_external_answer_quality_vs_tabu_search[0pct]` | 0,75 |
| Análisis (preguntas finales, revisión del docente) | — | 1,00 |
| **Total** | | **5,00** |

Los niveles de calidad solo cuentan si la respuesta es factible.

In [3]:
!pip install 'git+https://github.com/CarlosYazid/Tabu-Search-Class.git'

  Cloning https://github.com/CarlosYazid/Tabu-Search-Class.git to /tmp/pip-req-build-y40_hhnb
  Running command git clone --filter=blob:none --quiet https://github.com/CarlosYazid/Tabu-Search-Class.git /tmp/pip-req-build-y40_hhnb
  Resolved https://github.com/CarlosYazid/Tabu-Search-Class.git to commit 10599d9cbd7bcbc768d483a6ae566dbfe0776418
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for Tabu-Search-Class: filename=tabu_search_class-0.1.0-py3-none-any.whl size=5657 sha256=6ef5928858bd2226ded12540b2329a1b1cbe8db76ede8476bbc4a275d9673eb8
  Stored in directory: /tmp/pip-ephem-wheel-cache-g7xnjyab/wheels/a8/bf/21/939845e22379261b017a1c38e731d4231fdce0fa7d13660ea8
Successfully built Tabu-Search-Class


In [38]:
# @title Imports

import itertools
import json
import re
import xml.etree.ElementTree as ET
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.linalg import eigh
from scipy.spatial.distance import pdist, squareform
from scipy.special import factorial

from tabu_search_class import checks, utils

In [39]:
# @title Configuraciones

RAIZ = Path()

# ----- Visualización y experimentos -----
MOSTRAR_ITERACIONES = 12                   # iteraciones que se imprimen con todo detalle
CALCULAR_OPTIMO = True                     # calcular el óptimo real (programación dinámica)
TENENCIAS_EXPERIMENTO = [0, 1, 3, 7, 15, 30]
SEMILLAS_EXPERIMENTO = 20

In [40]:
# @title Parámetros de la búsqueda tabú (cámbialos y vuelve a ejecutar)
TENENCIA = 3           # iteraciones que un movimiento queda prohibido (tabú)
ITERACIONES = 25      # máximo de iteraciones de la búsqueda tabú
SIN_MEJORA = 60        # se detiene si pasan estas iteraciones sin mejorar la mejor solución
ASPIRACION = True      # permitir un movimiento tabú si da una solución mejor que la mejor conocida
SEMILLA = 3            # semilla del azar del algoritmo

## Los datos
Fíjate en que las matrices **no son simétricas** (ir de A a B no cuesta lo mismo que de B a A), por eso el sentido en que se recorre una ruta importa.

In [41]:
t = pd.read_parquet(RAIZ / "time.parquet").set_index("Unnamed: 0")
d = pd.read_parquet(RAIZ / "distance.parquet").set_index("Unnamed: 0")
dem = pd.read_parquet(RAIZ / "demand.parquet")
p = pd.read_parquet(RAIZ / "params.parquet").iloc[0]

NOMBRES = t.index.to_numpy()
T = t.to_numpy(dtype=np.int64)
DIST = d.to_numpy(dtype=float)
DEM = dem["Demand (units)"].to_numpy(dtype=np.int64)
K = int(p["Number of vehicles"])
Q = int(p["Vehicle capacity (each)"])
DEPOSITO = int(p["Depot index"])
COSTO_KM = float(p["Cost per km"])
COSTO_MIN = float(p["Cost per minute"])
data = {
    'names': NOMBRES,
    'num_vehicles': K,
    'depot': DEPOSITO,
    'time': T,
    'demand': DEM,
    'distance': DIST,
    'capacity': Q,
    'cost_per_km': COSTO_KM,
    'cost_per_min': COSTO_MIN,

}

CLIENTES = np.delete(np.arange(len(NOMBRES)), DEPOSITO)
CAN_CLIENTES = len(CLIENTES)
CORTOS = np.vectorize(lambda n: n.replace(" Health Center", ""))(NOMBRES)

In [ ]:
# @title Mapa esquemático

def coordenadas_mds(M):
    """Mapa 2D aproximado a partir de una matriz de distancias."""
    S = (M + M.T) / 2                                       # se simetriza para poder dibujarla
    n = len(S)
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ (S ** 2) @ J
    w, v = eigh(B, subset_by_index=[n - 2, n - 1])          # los 2 mayores valores propios
    coords = v * np.sqrt(np.maximum(w, 0))
    estres = np.linalg.norm(squareform(pdist(coords)) - S) / np.linalg.norm(S)
    return coords[:, ::-1], estres                          # el eje de mayor variación va en horizontal

COORDS, ESTRES = coordenadas_mds(DIST)

_, ax = plt.subplots(figsize=(10, 5))
ax.scatter(*COORDS[CLIENTES].T, s=DEM[CLIENTES] / 8, color="tab:blue", alpha=0.6, edgecolor="black")
ax.scatter(*COORDS[DEPOSITO], marker="*", s=400, color="gold", edgecolor="black", label="Depósito (MSO)")
for i in CLIENTES:
    ax.annotate(f"{i}. {CORTOS[i]}\n{DEM[i]:,}", COORDS[i], textcoords="offset points",
                xytext=(0, 13 if i % 2 else -25), ha="center", fontsize=7)
ax.set_title("Los centros de salud (el tamaño del círculo es la demanda)")
ax.margins(0.15)
ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal"); ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## El problema: decodificar y evaluar
Para no tener que lidiar con la capacidad dentro del vecindario, la solución `x` es solo un **orden** de los centros. La función `decodificar` lo corta en rutas consecutivas (algoritmo *Split*, programación dinámica) de la forma que **minimiza el tiempo total** sin pasarse de 6.000 unidades por vehículo. Así, **toda `x` produce una solución factible** y la búsqueda tabú solo se ocupa del orden.

La función de aptitud es `costo(x)` = tiempo total de las rutas decodificadas.

In [ ]:
def decodificar(x):
    """Split óptimo: corta el tour x en rutas consecutivas que respetan Q y minimizan el tiempo.
    Devuelve (tiempo total en min, rutas), con cada ruta como [depósito, ..., depósito]."""
    n = len(x)
    acum = np.concatenate(([0], np.cumsum(T[x[:-1], x[1:]])))   # acum[k]: tiempo de p[0] a p[k]
    carga = np.concatenate(([0], np.cumsum(DEM[x])))            # carga[k]: demanda de p[0..k-1]
    ida, vuelta = T[DEPOSITO, x], T[x, DEPOSITO]
    V = np.full(n + 1, np.inf)                                  # V[j]: mejor tiempo para atender p[0..j-1]
    V[0] = 0
    padre = np.zeros(n + 1, dtype=int)
    for j in range(1, n + 1):
        for i in range(j - 1, -1, -1):                          # la última ruta atiende p[i..j-1]
            if carga[j] - carga[i] > Q:
                break
            c = V[i] + ida[i] + (acum[j - 1] - acum[i]) + vuelta[j - 1]
            if c < V[j]:
                V[j], padre[j] = c, i
    rutas, j = [], n
    while j > 0:
        i = padre[j]
        rutas.append([DEPOSITO, *x[i:j].tolist(), DEPOSITO])
        j = i
    rutas.reverse()
    assert len(rutas) <= K, "se necesitarían más vehículos de los disponibles"
    return int(V[n]), rutas

def costo(x):
    """f(x): tiempo total (min) de la solución. Se guarda en caché porque se repiten muchas soluciones."""
    return decodificar(x)[0]

def solucion_aleatoria(rng):
    """Representación: x = permutación de los centros."""
    return rng.permutation(CLIENTES)

def aplicar(x, mov):
    """Devuelve el vecino de x que resulta de intercambiar las posiciones i y j."""
    i, j = mov
    y = x.copy()
    y[i], y[j] = y[j], y[i]
    return y

def reportar(rutas):
    """Tiempo (min), distancia (km) y costo operativo (0,8/km + 0,3/min) de unas rutas."""
    a = np.concatenate([r[:-1] for r in rutas])
    b = np.concatenate([r[1:] for r in rutas])
    minutos, km = int(T[a, b].sum()), float(DIST[a, b].sum())
    return minutos, km, COSTO_KM * km + COSTO_MIN * minutos

In [ ]:
# @title Graficas
def graficar_rutas(rutas, titulo="Rutas", ax=None):
    """Rutas sobre el mapa esquemático. Las flechas curvas muestran el sentido del recorrido."""
    solo = ax is None
    if solo:
        _, ax = plt.subplots(figsize=(9, 6))
    colores = plt.cm.tab10.colors
    for k, r in enumerate(rutas):
        c = colores[k % len(colores)]
        for a, b in zip(r[:-1], r[1:]):
            ax.annotate("", xy=COORDS[b], xytext=COORDS[a], zorder=2,
                        arrowprops=dict(arrowstyle="-|>", color=c, lw=2, shrinkA=9, shrinkB=9,
                                        connectionstyle="arc3,rad=0.15"))
        ax.plot([], [], color=c, lw=2,
                label=f"Ruta {k + 1}: {int(T[r[:-1], r[1:]].sum())} min, carga {int(DEM[r].sum()):,}")
    ax.scatter(*COORDS[CLIENTES].T, s=DEM[CLIENTES] / 15, color="lightgray", edgecolor="black", zorder=3)
    ax.scatter(*COORDS[DEPOSITO], marker="*", s=350, color="gold", edgecolor="black", zorder=4)
    for i in CLIENTES:
        ax.annotate(str(i), COORDS[i], ha="center", va="center", fontsize=8, zorder=5)
    total = reportar(rutas)[0]
    ax.set_title(f"{titulo}\n{len(rutas)} vehículos, {total} min", fontsize=10)
    ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False)
    ax.margins(0.08)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    if solo:
        plt.tight_layout()
        plt.show()

def graficar_cargas(rutas, titulo="Carga y tiempo por vehículo"):
    """Barras de carga (con la capacidad) y de tiempo de cada ruta."""
    cargas = [int(DEM[r].sum()) for r in rutas]
    tiempos = [int(T[r[:-1], r[1:]].sum()) for r in rutas]
    etiquetas = [f"Ruta {k + 1}\n" + "-".join(str(c) for c in r[1:-1]) for k, r in enumerate(rutas)]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
    a1.bar(etiquetas, cargas, color="tab:blue", alpha=0.8)
    a1.axhline(Q, color="red", linestyle="--", label=f"capacidad = {Q:,}")
    a1.set_ylim(0, Q * 1.2)
    a1.set_ylabel("unidades"); a1.set_title("Carga por vehículo"); a1.legend(fontsize=8)
    a2.bar(etiquetas, tiempos, color="tab:orange", alpha=0.8)
    a2.set_ylabel("minutos"); a2.set_title(f"Tiempo por vehículo (total = {sum(tiempos)} min)")
    for a in (a1, a2):
        a.tick_params(axis="x", labelsize=7)
    fig.suptitle(titulo, fontsize=11)
    plt.tight_layout()
    plt.show()

_, rutas_ejemplo = decodificar(CLIENTES)
graficar_rutas(rutas_ejemplo, "Ejemplo: tour 1-2-...-10 decodificado")
graficar_cargas(rutas_ejemplo, "Ejemplo: tour 1-2-...-10")

## Óptimo real (solo como referencia)

In [12]:
_, total_time, _, _ = utils.solve_with_tabu_search(data)
print("Óptimo real:", total_time, "min")

Óptimo real: 317 min


## Vecindario
La búsqueda local y la búsqueda tabú usan el **mismo** vecindario: intercambiar dos centros **consecutivos** del tour. Un movimiento es la pareja de posiciones `(i, j)` (aquí siempre `j = i + 1`).

**TODO 1.** Completa `vecindario(n)`.

In [13]:
def vecindario(n):
    """N(x): intercambiar dos centros CONSECUTIVOS del tour. Cada movimiento es la pareja (i, i + 1).
    Devuelve la lista con TODOS los movimientos posibles para un tour de n centros."""
    # ===================== TODO 1 =====================
    # Genera las parejas (i, i + 1) para i = 0, 1, ..., n - 2.
    # Pista: una lista por comprensión con range(n - 1).
    # ==================================================
    raise NotImplementedError("Completa el TODO 1")

In [ ]:
# @title Check del TODO 1
checks.test_vecindario(len(CLIENTES), vecindario, aplicar)

## Línea base: búsqueda local (*hill climbing*)
Solo acepta movimientos que **mejoran**, así que se detiene en el primer óptimo local. También hay una versión con **reinicios** que gasta el mismo esfuerzo (mismas evaluaciones) que la búsqueda tabú, pero sin memoria.

In [25]:
def busqueda_local(x):
    """Descenso: se mueve al mejor vecino mientras MEJORE. Termina en un óptimo local.
    Devuelve la solución, su costo, los pasos dados y las soluciones evaluadas."""
    fx = costo(x)
    pasos = evaluaciones = 0
    while True:
        mejor_y, mejor_fy = None, fx
        for mov in vecindario(len(x)):
            y = aplicar(x, mov)
            fy = costo(y)
            evaluaciones += 1
            if fy < mejor_fy:
                mejor_y, mejor_fy = y, fy
        if mejor_y is None:                           # ningún vecino mejora: óptimo local
            return x, fx, pasos, evaluaciones
        x, fx = mejor_y, mejor_fy
        pasos += 1

def reinicios(evaluaciones, rng):
    """Mismo esfuerzo que la búsqueda tabú, pero sin memoria: búsquedas locales desde tours
    aleatorios nuevos. Devuelve el mejor costo."""
    mejor, usadas = None, 0
    while usadas < evaluaciones:
        _, f, _, e = busqueda_local(solucion_aleatoria(rng))
        usadas += e
        if mejor is None or f < mejor:
            mejor = f
    return mejor

## Búsqueda tabú
En cada iteración: (1) evaluar todo el vecindario, (2) moverse al mejor vecino **permitido aunque empeore**, (3) actualizar la lista tabú, (4) actualizar la mejor solución y (5) revisar el criterio de parada.

La **lista tabú** es un diccionario `{(centro, posición): última iteración en la que sigue prohibido}`. Si en la iteración `it` se mueve el centro `a` desde la posición `i`, se guarda `(a, i) -> it + tenencia`: no se le deja volver a `i` hasta que pase ese tiempo.

**TODO 2.** Completa `es_tabu`. **TODO 3.** Completa `actualizar_lista_tabu`.

In [16]:
def es_tabu(x, mov, lista_tabu, it):
    """¿El movimiento (i, j) está prohibido en la iteración `it`?
    Está prohibido si dejaría a algún centro en una posición que aún es tabú."""
    # ===================== TODO 2 =====================
    # Si se intercambian las posiciones i y j, el centro a = x[i] pasa a la posición j
    # y el centro b = x[j] pasa a la posición i.
    # El movimiento es tabú si lista_tabu[(a, j)] o lista_tabu[(b, i)] sigue vigente,
    # es decir, si su valor guardado es >= it (usa .get(clave, 0) para las claves ausentes).
    # ==================================================
    raise NotImplementedError("Completa el TODO 2")

def actualizar_lista_tabu(x, mov, lista_tabu, it, tenencia):
    """Registra el movimiento (i, j) que se va a ejecutar sobre x (x es la solución ANTES de moverse).
    Prohíbe devolver cada centro intercambiado a la posición que acaba de dejar."""
    # ===================== TODO 3 =====================
    # Para los dos centros que cambian de posición, guarda en lista_tabu
    # la clave (centro, posición que deja) con el valor it + tenencia.
    # ==================================================
    raise NotImplementedError("Completa el TODO 3")

In [ ]:
# @title Checks de los TODO 2 y 3
checks.test_es_tabu_y_actualizar_tabu(CLIENTES, es_tabu, actualizar_lista_tabu, aplicar)

Ahora el ciclo completo. **TODO 4:** dentro del ciclo que recorre el vecindario, aplica el criterio de **aspiración** y guarda el mejor vecino permitido (con una lista de candidatos para resolver empates al azar).

In [18]:
def busqueda_tabu(tenencia, iteraciones, sin_mejora, aspiracion, rng, mostrar=False):
    x = solucion_aleatoria(rng)
    fx = costo(x)
    inicial = fx
    mejor_x, mejor_f = x[:], fx
    lista_tabu = {}                  # (centro, posición) -> última iteración en que sigue prohibido
    historial = {"actual": [fx], "mejor": [fx], "empeora": [], "aspiracion": []}
    visitadas = {tuple(x)}
    evaluaciones = 0
    ultima_mejora = it = 0

    if mostrar:
        print(f"Solución inicial (aleatoria): costo = {fx} min\n")

    for it in range(1, iteraciones + 1):
        # 1) Evaluar TODO el vecindario y quedarse con el mejor vecino permitido
        candidatos, mejor_fy = [], None
        for mov in vecindario(CAN_CLIENTES):
            fy = costo(aplicar(x, mov))
            evaluaciones += 1
            tabu = es_tabu(x, mov, lista_tabu, it)
            # ===================== TODO 4: aspiración y selección =====================
            # (a) Si el movimiento es tabú y NO cumple la aspiración
            #     (aspiracion está activa y fy < mejor_f), sáltalo con `continue`.
            # (b) Si no, es un candidato:
            #       - si mejor_fy es None o fy < mejor_fy:  candidatos = [(mov, tabu)] y mejor_fy = fy
            #       - si fy == mejor_fy:                    candidatos.append((mov, tabu))
            #     (los empates se resuelven al azar más abajo)
            # ==========================================================================
            raise NotImplementedError("Completa el TODO 4")
        if not candidatos:
            if mostrar:
                print(f"  iteración {it}: no hay movimientos permitidos, la búsqueda se bloquea "
                      f"(si pasa desde la iteración 1, revisa el TODO 4)")
            break

        # 2) Moverse a ese vecino AUNQUE sea peor que la solución actual
        mov, es_aspiracion = candidatos[rng.integers(len(candidatos))]      # empates al azar
        i, j = mov
        a, b = x[i], x[j]
        fx_anterior = fx

        # 3) Actualizar la lista tabú (con x ANTES de moverse) y ejecutar el movimiento
        actualizar_lista_tabu(x, mov, lista_tabu, it, tenencia)
        x = aplicar(x, mov)
        fx = mejor_fy

        # 4) Actualizar la mejor solución conocida
        if fx < mejor_f:
            mejor_x, mejor_f = x[:], fx
            ultima_mejora = it

        visitadas.add(tuple(x))
        historial["actual"].append(fx)
        historial["mejor"].append(mejor_f)
        if fx > fx_anterior:
            historial["empeora"].append(it)
        if es_aspiracion:
            historial["aspiracion"].append(it)

        if mostrar:
            if it <= MOSTRAR_ITERACIONES:
                nota = ""
                if es_aspiracion:
                    nota = "  <- ASPIRACIÓN: era tabú, pero da un nuevo mejor"
                elif fx > fx_anterior:
                    nota = "  <- EMPEORA: ningún vecino permitido mejora, igual se acepta"
                elif ultima_mejora == it:
                    nota = "  <- nuevo mejor"
                activos = [f"{CORTOS[c]}@{p}" for (c, p), hasta in lista_tabu.items() if hasta > it]
                print(f"  iteración {it:3d}: {CORTOS[a]} (pos {i}) <-> {CORTOS[b]} (pos {j})"
                      f" | costo {fx_anterior:4d} -> {fx:4d}{nota}")
                print(f"                  lista tabú ({len(activos)}): {activos}")
            elif ultima_mejora == it:
                print(f"  iteración {it:3d}: nuevo mejor -> costo = {mejor_f}")
            if it == MOSTRAR_ITERACIONES:
                print("  ... (desde aquí solo se muestran los nuevos mejores)")

        # 5) Criterio de parada
        if it - ultima_mejora >= sin_mejora:
            if mostrar:
                print(f"  {sin_mejora} iteraciones sin mejorar: se detiene en la iteración {it}")
            break

    historial["visitadas"] = len(visitadas)
    historial["evaluaciones"] = evaluaciones
    return mejor_x, mejor_f, inicial, it, historial

## Ejecutar y revisar el comportamiento
Se ejecutan la búsqueda tabú, la búsqueda local (desde **la misma solución inicial**) y la local con reinicios. Luego se dibujan las rutas, la convergencia y las cargas.

In [29]:
def graficar_convergencia(historial, optimo=None, titulo="Convergencia de la búsqueda tabú"):
    _, ax = plt.subplots(figsize=(9, 4))
    actual = np.array(historial["actual"])
    ax.plot(actual, label="costo de la solución actual", alpha=0.7)
    ax.plot(historial["mejor"], label="mejor costo encontrado", linewidth=2.5)
    emp = historial["empeora"]
    ax.scatter(emp, actual[emp], color="red", s=15, zorder=3, label="movimientos que empeoran (aceptados)")
    asp = historial["aspiracion"]
    if asp:
        ax.scatter(asp, actual[asp], marker="*", color="gold", edgecolor="black", s=180, zorder=4,
                   label="aspiración")
    if optimo is not None:
        ax.axhline(optimo, color="green", linestyle="--", label=f"óptimo real = {optimo}")
    tope = np.percentile(actual, 95)                  # se recortan los picos iniciales
    ax.set_ylim(min(historial["mejor"]) * 0.9 - 5, tope * 1.1 + 5)
    ax.set_xlabel("iteración")
    ax.set_ylabel("tiempo total (min)")
    ax.set_title(titulo)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# @title Reporte
print(f"Búsqueda tabú con TENENCIA = {TENENCIA}, ASPIRACION = {ASPIRACION}, SEMILLA = {SEMILLA}")
x_tabu, f_tabu, f_inicial, its, historial = busqueda_tabu(
    TENENCIA, ITERACIONES, SIN_MEJORA, ASPIRACION, np.random.default_rng(SEMILLA), mostrar=True)
x0 = solucion_aleatoria(np.random.default_rng(SEMILLA))              # la misma solución inicial
x_local, f_local, pasos, _ = busqueda_local(x0)
f_reinicios = reinicios(historial["evaluaciones"], np.random.default_rng(SEMILLA + 100))

print(f"\nSolución inicial (aleatoria):                    {f_inicial} min")
print(f"Búsqueda local desde la misma solución:          {f_local} min  (óptimo local tras {pasos} pasos)")
print(f"Búsqueda tabú:                                   {f_tabu} min  ({its} iteraciones)")
print(f"Búsqueda local con reinicios (mismo esfuerzo):   {f_reinicios} min")
if total_time is not None:
    print(f"Óptimo real:                                     {total_time} min")
    total = factorial(len(CLIENTES), exact=True)
    print(f"\nLa búsqueda tabú evaluó {historial['evaluaciones']:,} soluciones: "
          f"{100 * historial['evaluaciones'] / total:.2f}% de las {total:,} posibles.")
print(f"Aceptó {len(historial['empeora'])} movimientos que empeoraban, usó la aspiración "
      f"{len(historial['aspiracion'])} veces y visitó {historial['visitadas']} soluciones distintas.")
if not historial["empeora"] and total_time is not None and f_tabu > total_time:
    print("Pista: la búsqueda terminó sin llegar al óptimo y sin aceptar ningún movimiento que empeora. "
          "Prueba otra TENENCIA o revisa tu TODO 4.")

t_tabu, rutas_tabu = decodificar(x_tabu)
t_ini, rutas_ini = decodificar(x0)
t_loc, rutas_loc = decodificar(np.array(x_local))
print("\nRutas encontradas por la búsqueda tabú:")
for k, r in enumerate(rutas_tabu, 1):
    print(f"  Ruta {k}: {' -> '.join(CORTOS[i] for i in r)}  | carga {int(DEM[r].sum()):,}")
minutos, km, costo_op = reportar(rutas_tabu)
print(f"  Total: {minutos} min | {km:.1f} km | costo operativo {costo_op:.2f}")

In [ ]:
# @title Graficas

_, ejes = plt.subplots(1, 3, figsize=(18, 5.2))
graficar_rutas(rutas_ini, "Solución inicial (aleatoria)", ejes[0])
graficar_rutas(rutas_loc, "Búsqueda local: óptimo local", ejes[1])
graficar_rutas(rutas_tabu, "Búsqueda tabú", ejes[2])
plt.tight_layout()
plt.show()
graficar_convergencia(historial, total_time)
graficar_cargas(rutas_tabu, "Búsqueda tabú")

### Comparación con otras semillas
Un resultado aislado puede ser suerte. Se repite con 10 semillas y se usa una prueba estadística de Wilcoxon pareada para saber si la ventaja de la búsqueda tabú sobre la búsqueda local es real. Fíjate también en la columna de *reinicios*: es una línea base fuerte.

In [ ]:
tabu_c, local_c, reini_c = [], [], []
print("  semilla   tabú   local   reinicios")
for s in range(1, 11):
    _, f, _, _, h = busqueda_tabu(TENENCIA, ITERACIONES, SIN_MEJORA, ASPIRACION, np.random.default_rng(s))
    _, l, _, _ = busqueda_local(solucion_aleatoria(np.random.default_rng(s)))
    r = reinicios(h["evaluaciones"], np.random.default_rng(s + 100))
    tabu_c.append(f); local_c.append(l); reini_c.append(r)
    print(f"  {s:7d}   {f:4d}   {l:5d}   {r:9d}")
tabu_c, local_c, reini_c = map(np.array, (tabu_c, local_c, reini_c))
print(f"  promedio {tabu_c.mean():6.1f}  {local_c.mean():6.1f}  {reini_c.mean():10.1f}")

if not np.array_equal(tabu_c, local_c):
    p_valor = stats.wilcoxon(tabu_c, local_c, alternative="less").pvalue
    print(f"\nWilcoxon (H1: tabú < local): p = {p_valor:.4f} -> "
          + ("la diferencia es estadísticamente significativa (p < 0,05)" if p_valor < 0.05
             else "no hay evidencia suficiente de que tabú sea mejor"))

_, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([tabu_c, local_c, reini_c], showmeans=True)
ax.set_xticklabels(["tabú", "local", "local con\nreinicios"])
if total_time is not None:
    ax.axhline(total_time, color="green", linestyle="--", label=f"óptimo real = {total_time}")
    ax.legend(fontsize=8)
ax.set_ylabel("tiempo total (min)")
ax.set_title("10 semillas: tiempo total de cada método")
plt.tight_layout()
plt.show()

## Experimento: efecto de la tenencia tabú
Se ejecuta la búsqueda con cada valor de `TENENCIAS_EXPERIMENTO` y varias semillas. Con tenencia muy pequeña el algoritmo **cicla** (repite las mismas soluciones); con tenencia muy grande queda **demasiado restringido**: casi todos los movimientos son tabú, la búsqueda se bloquea y se detiene antes de tiempo. Se reporta el promedio con su **intervalo de confianza del 95 %** y una prueba de Mann-Whitney contra la tenencia 0 (sin memoria).

In [ ]:
def experimento_tenencia():
    print(f"{SEMILLAS_EXPERIMENTO} ejecuciones por tenencia"
          + (f" (óptimo real = {total_time})" if total_time is not None else ""))
    print("  tenencia   promedio   IC 95 %           en el óptimo   % distintas   iteraciones")
    costos, diversidad, iteraciones = {}, {}, {}
    for tenencia in TENENCIAS_EXPERIMENTO:
        c, d, n_it = [], [], []
        for s in range(1, SEMILLAS_EXPERIMENTO + 1):
            _, f, _, its, h = busqueda_tabu(tenencia, ITERACIONES, SIN_MEJORA, ASPIRACION,
                                            np.random.default_rng(s))
            c.append(f)
            d.append(h["visitadas"] / (its + 1))
            n_it.append(its)
        c = np.array(c)
        costos[tenencia], diversidad[tenencia], iteraciones[tenencia] = c, np.mean(d), np.mean(n_it)
        error = stats.sem(c)
        ic = stats.t.interval(0.95, len(c) - 1, loc=c.mean(), scale=error) if error > 0 else (c.mean(),) * 2
        aciertos = f"{int(np.sum(c == total_time)):2d} / {len(c)}" if total_time is not None else "-"
        print(f"  {tenencia:8d}   {c.mean():8.1f}   [{ic[0]:6.1f}, {ic[1]:6.1f}]   {aciertos:>12}"
              f"   {100 * diversidad[tenencia]:10.0f}%   {iteraciones[tenencia]:10.0f}")

    base = costos[TENENCIAS_EXPERIMENTO[0]]
    print(f"\nMann-Whitney contra tenencia = {TENENCIAS_EXPERIMENTO[0]} (H1: la otra tenencia da menor costo):")
    for tenencia in TENENCIAS_EXPERIMENTO[1:]:
        if np.array_equal(base, costos[tenencia]):
            print(f"  tenencia {tenencia:2d}: resultados idénticos")
            continue
        p = stats.mannwhitneyu(costos[tenencia], base, alternative="less").pvalue
        print(f"  tenencia {tenencia:2d}: p = {p:.4f} " + ("(mejora significativa)" if p < 0.05 else ""))

    _, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    a1.boxplot([costos[t] for t in TENENCIAS_EXPERIMENTO], showmeans=True)
    a1.set_xticklabels(TENENCIAS_EXPERIMENTO)
    if total_time is not None:
        a1.axhline(total_time, color="green", linestyle="--", label="óptimo real")
        a1.legend(fontsize=8)
    a1.set_xlabel("tenencia"); a1.set_ylabel("tiempo total (min)")
    a1.set_title(f"Costo final con {SEMILLAS_EXPERIMENTO} semillas")
    a2.plot(TENENCIAS_EXPERIMENTO, [100 * diversidad[t] for t in TENENCIAS_EXPERIMENTO], "o-", color="tab:blue")
    a2.set_xlabel("tenencia"); a2.set_ylabel("% de soluciones distintas visitadas", color="tab:blue")
    a3 = a2.twinx()
    a3.plot(TENENCIAS_EXPERIMENTO, [iteraciones[t] for t in TENENCIAS_EXPERIMENTO], "s--", color="tab:red")
    a3.set_ylabel("iteraciones antes de parar", color="tab:red")
    a2.set_title("Poca diversidad = cicla; pocas iteraciones = se bloquea")
    plt.tight_layout()
    plt.show()

    # Recorrido de la solución actual con tres tenencias distintas (misma semilla)
    elegidas = sorted({min(TENENCIAS_EXPERIMENTO), TENENCIA, max(TENENCIAS_EXPERIMENTO)})
    _, ejes = plt.subplots(1, len(elegidas), figsize=(4.5 * len(elegidas), 3.6), sharey=True, squeeze=False)
    for ax, tenencia in zip(ejes[0], elegidas):
        _, f, _, its, h = busqueda_tabu(tenencia, ITERACIONES, SIN_MEJORA, ASPIRACION,
                                        np.random.default_rng(SEMILLA))
        ax.plot(h["mejor"], linewidth=3, color="tab:orange", label="mejor")
        ax.plot(h["actual"], color="tab:blue", alpha=0.8, label="actual")
        if total_time is not None:
            ax.axhline(total_time, color="green", linestyle="--", label="óptimo")
        ax.set_title(f"tenencia = {tenencia}  (mejor = {f})\n{h['visitadas']} soluciones distintas "
                     f"en {its} iteraciones", fontsize=10)
        ax.set_xlabel("iteración")
    ejes[0][0].set_ylabel("tiempo total (min)")
    if total_time is not None:
        ejes[0][0].set_ylim(total_time * 0.96, total_time * 1.22)
    ejes[0][0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


experimento_tenencia()

## Entrega y evaluación automática
La celda siguiente toma las rutas de **tu** búsqueda tabú (`rutas_tabu`),
usando la mejor semilla de 100:

1. **Factibilidad de tu respuesta**: cada ruta sale y vuelve al depósito, cada centro se visita una sola vez, ninguna ruta pasa de 6.000 unidades y se usan ≤ 15 vehículos.

2. **Calidad**: tu tiempo total frente al de referencia (10 s de cómputo).

In [ ]:
# @title Coherencia de la respuesta
checks.test_external_answer_coherence(data, rutas_tabu)

In [ ]:
# @title Tiempo total a ≤ 10 % de la referencia
checks.test_external_answer_quality_vs_tabu_search(data, rutas_tabu, 0.1)

In [ ]:
# @title Tiempo total a ≤ 5 % de la referencia
checks.test_external_answer_quality_vs_tabu_search(data, rutas_tabu, 0.05)

In [ ]:
# @title Tiempo total a ≤ 2 % de la referencia
checks.test_external_answer_quality_vs_tabu_search(data, rutas_tabu, 0.02)

In [ ]:
# @title Tiempo total a ≤ 0 % de la referencia
checks.test_external_answer_quality_vs_tabu_search(data, rutas_tabu, 0.00000001)

## Análisis
Responde en un par de frases cada una. Apóyate en las gráficas y tablas de tu propia ejecución.

1. **Empeorar para mejorar.** En la gráfica de convergencia, ¿qué ocurre con la curva *actual* cuando la búsqueda acepta un movimiento que empeora? ¿Por qué la búsqueda local nunca puede hacerlo y qué pierde por eso?

2. **La tenencia.** Según el experimento (promedio, intervalo de confianza, p-valores y % de soluciones distintas), ¿qué tenencia recomiendas? Explica qué le pasa a la búsqueda con una tenencia de 0 y con una de 30.

3. Compara tu tiempo total con el de la referencia y con el óptimo real. Si no igualaste la referencia, ¿qué cambiarías (tenencia, iteraciones, aspiración) y por qué? Si la igualaste, ¿crees que es suerte de la semilla? Justifícalo con la comparación de semillas.

4. **Reinicios y vecindario.** En la comparación de semillas, la búsqueda local con reinicios rinde casi igual que la tabú. ¿Por qué crees que ocurre en un problema de este tamaño y cuándo esperarías que la tabú gane? Para comprobar el efecto del vecindario, cambia el TODO 1 para que devuelva *todas* las parejas `(i, j)` con `i < j` (45 vecinos), repite la comparación y cuenta qué pasa con la búsqueda local. (Deja el TODO 1 como estaba antes de entregar.)

5. **El sentido importa.** El tiempo de MSO a Al-Ghiran es 26 min y el de Al-Ghiran a MSO es 31 min. ¿Cómo afecta esto a que dos rutas con los mismos centros tengan tiempos distintos? Fíjate en el sentido de las flechas del mapa.

**Respuestas:**

*(escribe aquí tus respuestas)*